In [1]:
import numpy as np
import pandas as pd 
import plotly.express as px 
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

ModuleNotFoundError: No module named 'plotly'

In [ ]:
import json
from urllib.request import urlopen

with urlopen(
    "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
) as response:
    counties_geojson = json.load(response)

### Afordable Care Act Data for 2025 (ACA)

In [ ]:
aca = pd.read_csv("/Users/Katia/Desktop/2024_US_President_Elections_and_Public_Benefits_Ohio/aca_data_2025.csv")
aca['aca_enrollment'] = aca['aca_enrollment'].replace(",", "", regex=True).astype(int)
aca["fips_code"] = (aca["fips_code"].astype(int).astype(str).str.zfill(5))

### Medicaid & Medicare Dual Enrollment Data for 2023

In [ ]:
mme = pd.read_csv(
    "/Users/Katia/Desktop/2024_US_President_Elections_and_Public_Benefits_Ohio/medicaid_medicare_data_2023.csv"
).rename(columns = {
    'YEAR': 'year',
    'MONTH': 'month',
    'BENE_STATE_ABRVTN': 'state',
    'BENE_COUNTY_DESC': 'county',
    'BENE_FIPS_CD': 'fips_code',
    'DUAL_TOT_BENES': 'mm_enrollment'
})

mme['fips_code'] = mme["fips_code"].astype(int).astype(str).str.zfill(5)
mme['county'] = mme['county'].str.replace("County", "", regex=True).str.strip()
# Group by 'county' and sum the 'mm_enrollment'
mme = mme.groupby(['county', 'fips_code'])['mm_enrollment'].sum().reset_index()


### 2024 Presidential Elections Results Data 

In [ ]:
election_results = pd.read_excel(
    "/Users/Katia/Desktop/2024_US_President_Elections_and_Public_Benefits_Ohio/president_election_results_ohio_2024.xlsx",
    sheet_name="President and Vice President",
).drop(columns=["Region Name", "Media Market"], axis=1).rename(columns = {
    'County Name': 'county',
    'Registered Voters': 'reg_voters',
    'Ballots Counted': 'ballots_counted',
    'Official Voter Turnout': 'voter_turnout',
    'Kamala D. Harris and Tim Walz (D)': 'harris_votes',
    'Donald J. Trump and JD Vance (R)': 'trump_votes',
    'Winner': 'winner'
})

election_results["county"] = (
    election_results["county"].astype(str).str.strip()
)

# Identify counties where Harris and Trump won
election_results["winner"] = election_results.apply(
    lambda row: "Harris"
    if row["harris_votes"]
    > row["trump_votes"]
    else (
        "Trump"
        if row["trump_votes"]
        > row["harris_votes"]
        else "Tie"
    ),
    axis=1,
)

# Calculate the percent victory margin for each county
election_results["pct_diff"] = round(
    (
        election_results["harris_votes"]
        - election_results["trump_votes"]
    )
    / election_results["ballots_counted"],
    4,
)

### Ohio County Population Data (2024)

In [ ]:
county_popul = pd.read_excel(
    "/Users/Katia/Desktop/2024_US_President_Elections_and_Public_Benefits_Ohio/ohio_counties_population_2024.xlsx",
    header=None,
    names=["County Name", "Population"],
).drop([0, 1], axis=0).rename(columns = {
    'County Name': 'county',
    'Population': 'population'
})

county_popul["county"] = county_popul["county"].str.replace(
    r"^\.|\ County, Ohio$", "", regex=True
)
county_popul["population"] = pd.to_numeric(county_popul["population"])
county_popul = county_popul.reset_index(drop=True)

### Merge 4 Datasets

In [ ]:
# ==========================================
# 2. Merge Data, Normalize, and Restructure
# ==========================================

# 1. Start with your base dataframe (e.g., aca)
# 2. Left-merge the others sequentially based on the available key

# If mme and election_results use 'fips_code':
merged_df = (
    aca
    .merge(mme, on='fips_code', how='left')
    .merge(election_results, on='county', how='left')
)

# Note: Since county_popul currently uses 'county' instead of 'fips_code', 
# you can map it if one of your other datasets contains BOTH 'county' and 'fips_code'
# For example, if election_results has both:
merged_df = merged_df.merge(county_popul, on='county', how='left')

# 1. Drop the 'state' column
merged_df = merged_df.drop(columns=['state'])

# 2. Shift 'county' to be the first column
# This pulls 'county' out, and grabs all other columns dynamically
cols = ['county'] + [col for col in merged_df.columns if col != 'county']
merged_df = merged_df[cols]

### Summary

In [ ]:
# ==========================================
# 4. Final Aggregations
# ==========================================

summary_by_winner = (
    merged_df.groupby("winner")
    .agg(
        Population=("population", "sum"),
        ACA=("aca_enrollment", "sum"),
        MM=("mm_enrollment", "sum"),
    )
    .applymap(lambda x: f"{int(x):,}")
)

### Feature Engineering & Model Building

In [ ]:
# =========================================================================
# PART 1: First Analytical Steps (Feature Engineering & Scaling)
# =========================================================================

# Engineer rate columns for ACA & MM enrollments
merged_df['aca_rate'], merged_df['mm_rate'] = (merged_df['aca_enrollment']/merged_df['population']), (merged_df['mm_enrollment']/merged_df['population'])

# Step 1.1: Combine ACA and Medicaid/Medicare rates into a single Benefit Dependency feature
# (Using rates prevents population size from artificially skewing the unsupervised models)
merged_df["total_benefit_dependency"] = (
    merged_df["aca_rate"] + merged_df["mm_rate"]
)

# Step 1.2: Select the features to feed into the unsupervised models.
# We map political margin (pct_diff) against our engineered dependency feature.
features = ["pct_diff", "total_benefit_dependency"]
X = merged_df[features]

# Step 1.3: Scale the features.
# Distance-based models (LOF & One-Class SVM) require features to be on the same scale.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# =========================================================================
# PART 2: Build, Train, and Run the 3 Anomaly Detection Models
# =========================================================================

# We will set an expected contamination rate of ~8% (approx. 7 out of 88 Ohio counties)
contamination_rate = 0.08

# --- Model 1: Isolation Forest ---
iso_forest = IsolationForest(
    contamination=contamination_rate, random_state=42
)
# fit_predict returns 1 for inliers, -1 for statistical outliers
merged_df["iso_raw_outlier"] = iso_forest.fit_predict(X_scaled)

# --- Model 2: Local Outlier Factor (LOF) ---
# Note: LOF uses novelty=False by default for fit_predict on training data
lof = LocalOutlierFactor(n_neighbors=15, contamination=contamination_rate)
merged_df["lof_raw_outlier"] = lof.fit_predict(X_scaled)

# --- Model 3: One-Class SVM ---
# nu parameter acts as an upper bound on training errors (equivalent to contamination)
oc_svm = OneClassSVM(nu=contamination_rate, kernel="rbf", gamma="scale")
merged_df["svm_raw_outlier"] = oc_svm.fit_predict(X_scaled)

In [ ]:
# =========================================================================
# PART 3: Post-Processing & Outlier Labeling
# =========================================================================

def label_targeted_outliers(df, model_col):
    """
    Converts raw anomaly model output into a simple classification.

    - Targeted Outlier: County flagged as anomalous by the model
    - Normal County: County not flagged as anomalous
    """

    is_outlier = df[model_col] == -1

    return np.where(
        is_outlier,
        "Targeted Outlier",
        "Normal County"
    )


# Generate status labels for each model
merged_df["iso_status"] = label_targeted_outliers(
    merged_df, "iso_raw_outlier"
)

merged_df["lof_status"] = label_targeted_outliers(
    merged_df, "lof_raw_outlier"
)

merged_df["svm_status"] = label_targeted_outliers(
    merged_df, "svm_raw_outlier"
)

In [ ]:
# =========================================================================
# PART 4: Binary Outlier Flags
# =========================================================================

merged_df["iso_flag"] = (
    merged_df["iso_raw_outlier"] == -1
)

merged_df["lof_flag"] = (
    merged_df["lof_raw_outlier"] == -1
)

merged_df["svm_flag"] = (
    merged_df["svm_raw_outlier"] == -1
)

In [ ]:
# =========================================================================
# PART 5: Model Comparison Summary
# =========================================================================

model_summary = pd.DataFrame({
    "Model": [
        "Isolation Forest",
        "Local Outlier Factor",
        "One-Class SVM"
    ],
    "Outliers Detected": [
        merged_df["iso_flag"].sum(),
        merged_df["lof_flag"].sum(),
        merged_df["svm_flag"].sum()
    ]
})

In [ ]:
# =========================================================================
# PART 6: Consensus Outlier Framework
# =========================================================================
merged_df["consensus_score"] = (
    merged_df["iso_flag"].astype(int)
    + merged_df["lof_flag"].astype(int)
    + merged_df["svm_flag"].astype(int)
)

def classify_consensus(score):
    if score == 3:
        return "Consensus Outlier (All 3 Models)"
    elif score == 2:
        return "Majority Outlier (2 Models)"
    elif score == 1:
        return "Single Model Outlier"
    else:
        return "Normal County"

merged_df["consensus_status"] = (
    merged_df["consensus_score"]
    .apply(classify_consensus)
)

In [ ]:
# =========================================================================
# PART 7: Robust Outlier Counties
# =========================================================================

consensus_outliers = merged_df[
    merged_df["consensus_score"] >= 2
][[
    "county",
    "winner",
    "pct_diff",
    "total_benefit_dependency",
    "consensus_score"
]]

consensus_outliers = consensus_outliers.sort_values(
    by="consensus_score",
    ascending=False
)

In [ ]:
# =========================================================================
# PART 8: Agreement Matrix
# =========================================================================

agreement_table = merged_df[
    [
        "county",
        "winner",
        "iso_flag",
        "lof_flag",
        "svm_flag",
        "consensus_score"
    ]
].sort_values(
    "consensus_score",
    ascending=False
)

In [ ]:
# ==============================
# Part 9: Final Metrics Function
# ==============================

def get_model_metrics(flag_col):

    total_counties = len(merged_df)

    total_outliers = int(
        merged_df[flag_col].sum()
    )

    return {
        "Total Counties": total_counties,

        "Outliers Detected": total_outliers,

        "Outlier Rate (%)": round(
            total_outliers / total_counties * 100,
            1
        ),

        "Flagged Counties": ", ".join(
            merged_df.loc[
                merged_df[flag_col],
                "county"
            ].tolist()
        )
    }

### Visualizations and Summaries

In [ ]:
# Slide 3 - Healthcare Dependency Across Ohio
fig1 = px.choropleth(
    merged_df,
    geojson=counties_geojson,
    locations="fips_code",
    color="total_benefit_dependency",
    color_continuous_scale="Reds",
    scope="usa"
)

fig1.update_geos(
    fitbounds="locations",
    visible=False,
    framewidth=0,
    framecolor="white"
)

fig1.update_layout(
    # Use standard presentation aspect ratio or match your export dimensions
    width=900,
    height=700,
    margin=dict(l=20, r=20, t=70, b=20), # Generous margins to let the map "breathe"
    paper_bgcolor="white",
    plot_bgcolor="white",
    geo=dict(bgcolor="rgba(0,0,0,0)"),
    
    # Bold, clean title typography optimized for slides
    title=dict(
        text="<b>Healthcare Dependency Across Ohio Counties</b>",
        x=0.05, # Left-aligned like premium editorial charts
        y=0.95,
        font=dict(
            family="Arial, Helvetica, sans-serif",
            size=24,
            color="#212529" # Off-black for a softer look
        )
    ),
    
    # Modern, horizontal colorbar placed cleanly underneath the map
    coloraxis_colorbar=dict(
        title=dict(
            text="<b>Dependency Rate</b>",
            font=dict(family="Arial, sans-serif", size=14, color="black")
        ),
        thickness=15,
        len=0.5, # Takes up half the bottom width
        x=0.5,   # Centered
        y=-0.05,  # Placed below the map
        xanchor="center",
        orientation="h",
        tickfont=dict(family="Arial, sans-serif", size=12, color="black")
    )
)

In [ ]:
# Slide 4 — Election Margin vs Healthcare Dependency
fig2 = px.scatter(
    merged_df,
    x="pct_diff",
    y="total_benefit_dependency",
    color="winner",
    size="population",
    hover_name="county",
    # Lean color mapping for a cleaner visual split if needed (e.g., matching party colors)
    color_discrete_map={"Democrat": "#2B6CB0", "Republican": "#C53030"}
)

fig2.update_traces(
    marker=dict(
        line=dict(width=0.8, color="#FFFFFF"), # Clean white border looks less harsh than black
        opacity=0.85
    )
)

fig2.update_geos(
    fitbounds="locations",
    visible=False,
    projection_scale=8
)

# Presentation layout styling
fig2.update_layout(
    width=900,
    height=700,
    margin=dict(l=20, r=20, t=70, b=20),
    paper_bgcolor="white",
    plot_bgcolor="white",
    
    # Premium title typography
    title=dict(
        text="<b>Election Margin vs Healthcare Dependency</b>",
        x=0.05,
        y=0.95,
        font=dict(family="Arial, Helvetica, sans-serif", size=24, color="black")
    ),
    
    # Subdued gridlines that don't compete with the bubble sizes
    xaxis=dict(
        title="<b>Election Margin (%)</b>",
        titlefont=dict(family="Arial, sans-serif", size=14, color="black"),
        tickfont=dict(family="Arial, sans-serif", size=12, color="black"),
        showgrid=False,
        gridcolor="#E9ECEF"
    ),
    yaxis=dict(
        title="<b>Total Benefit Dependency</b>",
        titlefont=dict(family="Arial, sans-serif", size=14, color="black"),
        tickfont=dict(family="Arial, sans-serif", size=12, color="black"),
        showgrid=False,
        gridcolor="#E9ECEF"
    ),
    
    # Position categorical legend subtly at the top right
    legend=dict(
        title=dict(text="<b>Winner</b>", font=dict(family="Arial, sans-serif", size=12, color="black")),
        font=dict(family="Arial, sans-serif", size=12, color="black"),
        yanchor="top",
        y=0.98,
        xanchor="right",
        x=0.98,
        bgcolor="rgba(255,255,255,0.6)"
    )
)

In [ ]:
# Slide 5 — Isolation Forest Results
fig3 = px.scatter(
    merged_df,
    x="pct_diff",
    y="total_benefit_dependency",
    color="iso_status",
    size="population",
    text=np.where(
        merged_df["iso_flag"],
        merged_df["county"],
        ""
    ),
    # Optional styling contrast for Outliers vs Normal
    color_discrete_map={"Normal": "#CED4DA", "Outlier": "#DC3545"}
)

# Text and marker refinement
fig3.update_traces(
    textposition="top center",
    textfont=dict(family="Arial, sans-serif", size=11, color="#212529"),
    marker=dict(line=dict(width=0.8, color="#FFFFFF"), opacity=0.85)
)

fig3.update_geos(
    fitbounds="locations",
    visible=False,
    projection_scale=8
)

fig3.update_layout(
    width=900,
    height=700,
    margin=dict(l=20, r=20, t=70, b=20),
    paper_bgcolor="white",
    plot_bgcolor="white",
    
    title=dict(
        text="<b>Isolation Forest Outlier Detection</b>",
        x=0.05,
        y=0.95,
        font=dict(family="Arial, Helvetica, sans-serif", size=24, color="black")
    ),
    
    xaxis=dict(
        title="<b>Election Margin (%)</b>",
        titlefont=dict(family="Arial, sans-serif", size=14, color="black"),
        tickfont=dict(family="Arial, sans-serif", size=12, color="black"),
        showgrid=False,
        gridcolor="#E9ECEF"
    ),
    yaxis=dict(
        title="<b>Total Benefit Dependency</b>",
        titlefont=dict(family="Arial, sans-serif", size=14, color="black"),
        tickfont=dict(family="Arial, sans-serif", size=12, color="black"),
        showgrid=False
    ),
    
    legend=dict(
        title=dict(text="<b>Status</b>", font=dict(family="Arial, sans-serif", size=12, color="black")),
        font=dict(family="Arial, sans-serif", size=12, color="black"),
        yanchor="top",
        y=0.98,
        xanchor="right",
        x=0.98,
        bgcolor="white"
    )

In [ ]:
# Slide 6 — Model Comparison
model_summary = pd.DataFrame({
    "Model": [
        "Isolation Forest",
        "LOF",
        "One-Class SVM"
    ],
    "Outliers Detected": [
        merged_df["iso_flag"].sum(),
        merged_df["lof_flag"].sum(),
        merged_df["svm_flag"].sum()
    ]
})

In [ ]:
# Slide 7 — Consensus Map

# Color scheme
color_map = {
    "Normal County": "#D9D9D9",
    "Single Model Outlier": "#5BC0DE",
    "Majority Outlier (2 Models)": "#F0AD4E",
    "Consensus Outlier (All 3 Models)": "#D9534F"
}

# Create map
fig4 = px.choropleth(
    merged_df,
    geojson=counties_geojson,
    locations="fips_code",
    color="consensus_status",
    color_discrete_map=color_map,
    scope="usa"
)

fig4.update_geos(
    fitbounds="locations",
    visible=False,
    framewidth=0,
    framecolor="white"
)

fig4.update_layout(
    width=900,
    height=700,
    # Switched to transparent background to prevent white-box edges on slide templates
    paper_bgcolor="white",
    geo=dict(bgcolor="white"),
    
    margin=dict(l=20, r=20, t=70, b=20), 
    
    title=dict(
        text="<b>Ohio Counties Identified as Anomalous Across Multiple Models</b>",
        x=0.05,
        y=0.95,
        font=dict(family="Arial, Helvetica, sans-serif", size=22, color="black")
    ),
    
    legend_title_text="",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.05, # Positioned cleanly right below the map frame
        xanchor="center",
        x=0.5,
        font=dict(family="Arial, sans-serif", size=12, color="black")
    )
)

In [ ]:
# Slide 8 — Consensus Outlier Table
consensus_outliers = merged_df[
    merged_df["consensus_score"] >= 1
][[
    "county",
    "winner",
    "pct_diff",
    "total_benefit_dependency",
    "consensus_score"
]]

consensus_outliers = (
    consensus_outliers
    .sort_values(
        "consensus_score",
        ascending=False
    )
)